# 480 — Neurosynth meta-analytic regions

Assigns every electrode to **Neurosynth** association-test regions (the `neurosynth/`
folder: auditory · visual · motor control · phonological · semantic · lexical) by
sampling each FDR z-map at the electrode's fsaverage location (MNI305 → MNI152).
**Multi-label** — an electrode joins *every* region whose map is significant (z > `Z_THR`)
at its location. For each region it averages the member electrodes' concatenated
`[audio | picture | reading]` ERSP into a card. Exports to `outputs/pooling/neurosynth/`
(`neurosynth_labels.csv` + `neurosynth_info.json` + `region_cards/`), which the POOL web
page shows as a **'Neurosynth'** colour mode alongside Yeo.


In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


INPUT_DIR : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY | exists: True
OUTPUTS   : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\04_FBM_Pooling\outputs\pooling
COORDS    : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\coords | exists: True
conditions: ('audio', 'picture', 'reading') | zones: ('perception', 'pre_articulation', 'audio')
feature sets: ('hg', 'bands15') | window shapes: ('boxcar', 'gaussian')


In [2]:
# ---- knobs ----
GRID  = 'full'
Z_THR = 0.0            # 'inside' a region = its FDR z-map > this at the electrode (0 = any significant voxel)


## 1 — Assign electrodes to neurosynth regions, average, export


In [ ]:
coords = P.load_coords()
# restrict to the POOLED contacts (contacts_pool.csv from 460) so region counts match the analysis
out = P.export_neurosynth(INPUT_DIR, coords, P.OUTPUTS_ROOT / 'neurosynth', grid=GRID, z_thr=Z_THR,
                          pool_csv=P.OUTPUTS_ROOT / 'pool_web' / 'contacts_pool.csv')
print('neurosynth ->', out)


[lf_pool] neurosynth: restricted to 2691 pooled contacts
[lf_pool] neurosynth maps: ['auditory', 'broca', 'lexical', 'motor control', 'phonological', 'recall', 'semantic', 'visual']
[lf_pool] neurosynth: 2691 contacts, 800 in >=1 region (z>0.0)


## 2 — Membership summary


In [ ]:
import json, pandas as pd
info = json.load(open(P.OUTPUTS_ROOT / 'neurosynth' / 'neurosynth_info.json'))
display(pd.DataFrame([{'region': k, 'n_contacts': v['n'], 'n_ersp': v.get('n_ersp', 0)} for k, v in info.items()]))
lab = pd.read_csv(P.OUTPUTS_ROOT / 'neurosynth' / 'neurosynth_labels.csv')
print('contacts in >=1 region:', int((lab['neurosynth_primary'].fillna('') != '').sum()), '/', len(lab))
lab.head()
